In [13]:
import os
import holidays

import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb

from datetime import timedelta
from sklearn.metrics import mean_squared_error, mean_absolute_error

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
drive_path = '/content/drive/My Drive/Colab Notebooks/지역별 상품 물동량 통계 데이터'
data_name = 'seoulgu.parquet'

In [23]:
# data load
lazy_df = pl.scan_parquet(os.path.join(drive_path, data_name))

In [24]:
all_cols = lazy_df.columns
cat_cols = [c for c in all_cols if c.startswith('CAT_VOL_')]

/tmp/ipykernel_937/2883817882.py:1: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  all_cols = lazy_df.columns


In [ ]:
# 날짜 타입 변환 및 11개 카테고리 수평 합산 (total_vol)
lazy_processed = lazy_df.with_columns([
    pl.col('DELIVERY_YMD').cast(pl.Date).alias('DELIVERY_YMD'),
    pl.sum_horizontal(cat_cols).alias('total_vol') ])


In [ ]:
# (2) 자치구 및 일자별 집계 및 정렬
lazy_daily = (
    lazy_processed.group_by(['DELIVERY_YMD', 'RECEIVER_SIGUNGU_NM']).agg(
        pl.col('total_vol').sum()
        ).sort(['RECEIVER_SIGUNGU_NM', 'DELIVERY_YMD'])
    )


In [ ]:
# (3) 시계열 피처 생성 (캘린더 + 시차 Lag + 이동 통계량 Rolling)
lags = [7, 14, 21, 28]
windows = [7, 14, 28]

# Lag 피처 표현식
lag_exprs = [
    pl.col('total_vol').shift(lag).over('RECEIVER_SIGUNGU_NM').alias(f'vol_lag_{lag}')
    for lag in lags
]

# Rolling 피처 표현식 (Data Leakage 방지를 위해 Shift 7 적용)
rolling_exprs = []
for w in windows:
    rolling_exprs.append(
        pl.col('total_vol').shift(7).rolling_mean(window_size=w)
        .over('RECEIVER_SIGUNGU_NM').alias(f'rolling_mean_{w}'))
    rolling_exprs.append(
        pl.col('total_vol').shift(7).rolling_std(window_size=w)
        .over('RECEIVER_SIGUNGU_NM').alias(f'rolling_std_{w}'))


In [18]:
# 1. 자치구별 정적 프로파일 & 학습 가중치 생성 모듈
def compute_district_profiles(lazy_df: pl.LazyFrame) -> pl.LazyFrame:
    """
    [전략 4 & 5] 자치구별 품목(식품/패션) 비중 및 학습 가중치(sample_weight)를 산출합니다.
    """
    all_cols = lazy_df.columns
    cat_cols = [c for c in all_cols if c.startswith('CAT_VOL_')]

    # 일별 자치구별 합계 및 카테고리 집계
    lazy_daily = (
        lazy_df
        .with_columns([
            pl.col('DELIVERY_YMD').cast(pl.Date).alias('DELIVERY_YMD'),
            pl.sum_horizontal(cat_cols).alias('total_vol')
        ])
        .group_by(['DELIVERY_YMD', 'RECEIVER_SIGUNGU_NM'])
        .agg([
            pl.col('total_vol').sum(),
            pl.col('CAT_VOL_FOOD').sum().alias('vol_food'),
            pl.col('CAT_VOL_FASHION_CLOTHING').sum().alias('vol_fashion'),
            pl.col('CAT_VOL_BOOKS_MUSIC').sum().alias('vol_bookmusic'),
            pl.col('CAT_VOL_DIGITAL_APPLIANCES').sum().alias('vol_digital'),
        ])
        .sort(['RECEIVER_SIGUNGU_NM', 'DELIVERY_YMD'])
    )

    # 자치구별 품목 비중 (Food / Fashion Ratio) + 디지털, 도서/음반
    dist_profiles = lazy_daily.group_by('RECEIVER_SIGUNGU_NM').agg([
        (pl.col('vol_food').sum() / (pl.col('total_vol').sum() + 1e-5)).alias('food_ratio'),
        (pl.col('vol_fashion').sum() / (pl.col('total_vol').sum() + 1e-5)).alias('fashion_ratio'),
        (pl.col('vol_bookmusic').sum() / (pl.col('total_vol').sum() + 1e-5)).alias('bookmusic_ratio'),
        (pl.col('vol_digital').sum() / (pl.col('total_vol').sum() + 1e-5)).alias('digital_ratio')
    ])

    # 자치구별 학습 가중치 (Sample Weight)
    dist_weights = lazy_daily.group_by('RECEIVER_SIGUNGU_NM').agg(
        pl.col('total_vol').mean().alias('district_avg_vol')
    ).with_columns(
        (pl.col('district_avg_vol') / pl.col('district_avg_vol').mean()).alias('sample_weight')
    )

    # 프로파일 및 가중치 결합
    df_base = lazy_daily.join(dist_profiles, on='RECEIVER_SIGUNGU_NM', how='left')
    df_base = df_base.join(dist_weights.select(['RECEIVER_SIGUNGU_NM', 'sample_weight']), on='RECEIVER_SIGUNGU_NM', how='left')

    return df_base


# 2. 공휴일 전후 이월/쏠림 피처 생성
def create_holiday_features(lazy_df: pl.LazyFrame, start_year: int = 2019, end_year: int = 2023) -> pl.LazyFrame:
    """
    대한민국 공휴일 및 연휴 직전(D-1, D-2), 직후(D+1, D+2) 이월/쏠림 피처를 생성합니다.
    """
    kr_holidays = list(holidays.KR(years=range(start_year, end_year + 1)).keys())

    # (1) 휴무일 판별 (일요일 또는 법정 공휴일)
    df_feat = lazy_df.with_columns([
        (
            (pl.col('DELIVERY_YMD').dt.weekday() == 7) |
            (pl.col('DELIVERY_YMD').is_in(kr_holidays))
        ).cast(pl.Int64).alias('is_off_day')
    ])

    # (2) 연속 휴무 구간 ID 식별
    df_feat = df_feat.with_columns([
        (pl.col('is_off_day') != pl.col('is_off_day').shift(1)).fill_null(True).cum_sum().alias('off_block_id')
    ])

    # (3) 연속 휴무 일수 산출
    off_stats = (
        df_feat.filter(pl.col('is_off_day') == 1)
        .group_by('off_block_id')
        .agg(pl.col('is_off_day').sum().alias('consecutive_off_days'))
    )

    # (4) 전후 연휴 일수 및 피처 6종 생성
    df_final = (
        df_feat.join(off_stats, on='off_block_id', how='left')
        .with_columns(pl.col('consecutive_off_days').fill_null(0))
        .with_columns([
            pl.col('consecutive_off_days').shift(1).over('RECEIVER_SIGUNGU_NM').fill_null(0).alias('prev_off_days'),
            pl.col('consecutive_off_days').shift(-1).over('RECEIVER_SIGUNGU_NM').fill_null(0).alias('next_off_days')
        ])
        .with_columns([
            # Post-Holiday (연휴 직후 이월)
            ((pl.col('is_off_day') == 0) & (pl.col('prev_off_days') > 0)).cast(pl.Int64).alias('is_post_holiday_d1'),
            ((pl.col('is_off_day') == 0) & (pl.col('prev_off_days').shift(1).over('RECEIVER_SIGUNGU_NM') > 0)).cast(pl.Int64).alias('is_post_holiday_d2'),
            pl.when((pl.col('is_off_day') == 0) & (pl.col('prev_off_days') > 0)).then(pl.col('prev_off_days')).otherwise(0).alias('backlog_impact_score'),

            # Pre-Holiday (연휴 직전 쏠림)
            ((pl.col('is_off_day') == 0) & (pl.col('next_off_days') > 0)).cast(pl.Int64).alias('is_pre_holiday_d1'),
            ((pl.col('is_off_day') == 0) & (pl.col('next_off_days').shift(-1).over('RECEIVER_SIGUNGU_NM') > 0)).cast(pl.Int64).alias('is_pre_holiday_d2'),
            pl.when((pl.col('is_off_day') == 0) & (pl.col('next_off_days') > 0)).then(pl.col('next_off_days')).otherwise(0).alias('pre_holiday_impact_score')
        ])
    )
    return df_final


# 3. 캘린더 & 교차 변수 생성 모듈
def create_calendar_and_interaction_features(lazy_df: pl.LazyFrame) -> pl.LazyFrame:
    """
    [전략 3] 캘린더 변수 및 공휴일 x 요일 교차(Interaction) 피처를 생성합니다.
    """
    return lazy_df.with_columns([
        pl.col('DELIVERY_YMD').dt.year().alias('year'),
        pl.col('DELIVERY_YMD').dt.month().alias('month'),
        pl.col('DELIVERY_YMD').dt.day().alias('day'),
        pl.col('DELIVERY_YMD').dt.weekday().alias('dayofweek'),
        (pl.col('DELIVERY_YMD').dt.day() == 1).cast(pl.Int64).alias('is_month_start'),
        (pl.col('DELIVERY_YMD') == pl.col('DELIVERY_YMD').dt.month_end()).cast(pl.Int64).alias('is_month_end'),

        # 교차 피처
        (pl.col('is_post_holiday_d1') * pl.col('DELIVERY_YMD').dt.weekday()).alias('post_holiday_dow_inter'),
        (pl.col('is_pre_holiday_d1') * pl.col('pre_holiday_impact_score')).alias('pre_holiday_score_inter'),

        pl.col('RECEIVER_SIGUNGU_NM').cast(pl.Categorical)
    ])


# 4. 공통 파이프라인 조립 모듈 (Base Pipeline Assembler)
def build_base_pipeline(lazy_df: pl.LazyFrame) -> pl.LazyFrame:
    """
    1~3단계 기능을 조립하여 공통 기초 데이터 프레임 쿼리를 완성합니다.
    """
    df_profile = compute_district_profiles(lazy_df)
    df_holidays = create_holiday_features(df_profile)
    df_base = create_calendar_and_interaction_features(df_holidays)
    return df_base


# 5. t+step 시점별 동적 피처 생성 모듈
def create_step_specific_features(df_base: pl.DataFrame, step: int) -> pl.DataFrame:
    """
    [전략 1 & 2] 예측 대상 일차(t+step)에 맞춘 Lag, Rolling, 및 요일 대비 Ratio 피처를 생성합니다.
    """
    lags = [step, step + 7, step + 14, step + 21]

    # (1) Lag 피처 (step 기준 최신 시차 적용)
    lag_exprs = [
        pl.col('total_vol').shift(l).over('RECEIVER_SIGUNGU_NM').alias(f'vol_lag_{l}')
        for l in lags
    ]

    # (2) Rolling 피처 (step 시점 shift)
    rolling_exprs = [
        pl.col('total_vol').shift(step).rolling_mean(7).over('RECEIVER_SIGUNGU_NM').alias(f'rolling_mean_7_s{step}'),
        pl.col('total_vol').shift(step).rolling_std(7).over('RECEIVER_SIGUNGU_NM').alias(f'rolling_std_7_s{step}'),
        pl.col('total_vol').shift(step).rolling_mean(28).over('RECEIVER_SIGUNGU_NM').alias(f'rolling_mean_28_s{step}'),
        pl.col('total_vol').shift(step).rolling_std(28).over('RECEIVER_SIGUNGU_NM').alias(f'rolling_std_28_s{step}')
    ]

    # (3) [전략 2] 요일/추세 상대 비율 피처 (Ratio Features)
    ratio_exprs = [
        (pl.col('total_vol').shift(step).over('RECEIVER_SIGUNGU_NM') / (pl.col('total_vol').shift(step).rolling_mean(28).over('RECEIVER_SIGUNGU_NM') + 1e-5)).alias(f'ratio_lag_to_m28_s{step}'),
        (pl.col('total_vol').shift(step).over('RECEIVER_SIGUNGU_NM') / (pl.col('total_vol').shift(step + 7).over('RECEIVER_SIGUNGU_NM') + 1e-5)).alias(f'ratio_lag_wow_s{step}')
    ]

    return df_base.with_columns(lag_exprs + rolling_exprs + ratio_exprs).drop_nulls()


# 6. Step별 모델 학습 & 백테스팅 평가 모듈
def train_and_eval_step_model(step_df: pl.DataFrame, step: int, split_date: str):
    """
    [전략 5] t+step 전용 모델을 sample_weight와 np.log1p 기반으로 학습하고 백테스팅 오차를 산출합니다.
    """
    features = [
        'RECEIVER_SIGUNGU_NM', 'month', 'day', 'dayofweek', 'is_off_day',
        'is_month_start', 'is_month_end', 'food_ratio', 'fashion_ratio',
        'is_pre_holiday_d1', 'is_pre_holiday_d2', 'pre_holiday_impact_score', 'pre_holiday_score_inter',
        'is_post_holiday_d1', 'is_post_holiday_d2', 'backlog_impact_score', 'post_holiday_dow_inter',
        f'vol_lag_{step}', f'vol_lag_{step+7}', f'vol_lag_{step+14}', f'vol_lag_{step+21}',
        f'rolling_mean_7_s{step}', f'rolling_std_7_s{step}',
        f'rolling_mean_28_s{step}', f'rolling_std_28_s{step}',
        f'ratio_lag_to_m28_s{step}', f'ratio_lag_wow_s{step}'
    ]

    train_data = step_df.filter(pl.col('DELIVERY_YMD') < pl.lit(split_date).str.to_date())
    test_data = step_df.filter(pl.col('DELIVERY_YMD') >= pl.lit(split_date).str.to_date())

    X_tr = train_data.select(features).to_pandas()
    y_tr_log = np.log1p(train_data.select('total_vol').to_pandas().squeeze())
    w_tr = train_data.select('sample_weight').to_pandas().squeeze()

    X_te = test_data.select(features).to_pandas()
    y_te_raw = test_data.select('total_vol').to_pandas().squeeze()
    y_te_log = np.log1p(y_te_raw)

    model = lgb.LGBMRegressor(
        objective='regression',
        metric='rmse',
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=31,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_tr, y_tr_log,
        sample_weight=w_tr,
        eval_set=[(X_te, y_te_log)],
        categorical_feature=['RECEIVER_SIGUNGU_NM'],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )

    preds_log = model.predict(X_te)
    preds_raw = np.expm1(preds_log)

    wape = (np.sum(np.abs(y_te_raw - preds_raw)) / np.sum(y_te_raw)) * 100
    mape = np.mean(np.abs((y_te_raw - preds_raw) / y_te_raw)) * 100

    return model, {
        'horizon': f't+{step}',
        'WAPE': round(float(wape), 2),
        'MAPE': round(float(mape), 2)
    }


# 7. 전체 파이프라인 메인 오케스트레이터 (Runner)
def run_direct_multistep_forecasting(lazy_df: pl.LazyFrame, forecast_horizon: int = 7):
    """
    모든 모듈식 함수를 호출하여 Direct Multi-Step 파이프라인 전체를 제어 및 실행합니다.
    """
    print("1. 공통 기초 데이터베이스 피처 생성 중 (.collect())...")
    lazy_base = build_base_pipeline(lazy_df)
    df_base = lazy_base.collect()

    max_date_val = df_base['DELIVERY_YMD'].max()
    split_date = (max_date_val - timedelta(days=60)).strftime('%Y-%m-%d')

    models = {}
    metrics_summary = []

    print(f"2. t+1 ~ t+{forecast_horizon} 개별 모델 분리 학습 실행...")
    for step in range(1, forecast_horizon + 1):
        step_df = create_step_specific_features(df_base, step)
        model, metrics = train_and_eval_step_model(step_df, step, split_date)

        models[f't+{step}'] = model
        metrics_summary.append(metrics)
        print(f" -> [{metrics['horizon']}] WAPE: {metrics['WAPE']}% | MAPE: {metrics['MAPE']}%")

    return models, metrics_summary

In [26]:
trained_models, summary_metrics = run_direct_multistep_forecasting(lazy_df, forecast_horizon=7)

1. 공통 기초 데이터베이스 피처 생성 중 (.collect())...


/tmp/ipykernel_937/1859263256.py:6: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  all_cols = lazy_df.columns


2. t+1 ~ t+7 개별 모델 분리 학습 실행...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008146 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2834
[LightGBM] [Info] Number of data points in the train set: 43404, number of used features: 27
[LightGBM] [Info] Start training from score 10.091383
 -> [t+1] WAPE: 10.68% | MAPE: 23.97%
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007887 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2834
[LightGBM] [Info] Number of data points in the train set: 43379, number of used features: 27
[LightGBM] [Info] Start training from score 10.091336
 -> [t+2] WAPE: 10.59% | MAPE: 23.78%
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testi

In [36]:
summary_metrics

[{'horizon': 't+1', 'WAPE': 10.68, 'MAPE': 23.97},
 {'horizon': 't+2', 'WAPE': 10.59, 'MAPE': 23.78},
 {'horizon': 't+3', 'WAPE': 12.66, 'MAPE': 25.39},
 {'horizon': 't+4', 'WAPE': 12.88, 'MAPE': 25.54},
 {'horizon': 't+5', 'WAPE': 12.34, 'MAPE': 25.18},
 {'horizon': 't+6', 'WAPE': 11.92, 'MAPE': 24.86},
 {'horizon': 't+7', 'WAPE': 12.48, 'MAPE': 25.37}]

In [41]:
total_importance = []

for _model in trained_models:
    if not total_importance:
        df_importance = pd.DataFrame({
            'feature': trained_models[_model].feature_name_,
            f'split_{_model}': trained_models[_model].booster_.feature_importance(importance_type='split'),
            f'gain_{_model}': trained_models[_model].booster_.feature_importance(importance_type='gain')
        })
        total_importance.append(df_importance)
        continue
    df_importance = pd.DataFrame({
        f'split_{_model}': trained_models[_model].booster_.feature_importance(importance_type='split'),
        f'gain_{_model}': trained_models[_model].booster_.feature_importance(importance_type='gain')
    })
    total_importance.append(df_importance)




In [43]:
total_importance = pd.concat(total_importance, axis=1)
total_importance.sort_values(by=[col for col in total_importance.columns if 'gain' in col], ascending=False)

,feature,split_t+1,gain_t+1,split_t+2,gain_t+2,split_t+3,gain_t+3,split_t+4,gain_t+4,split_t+5,gain_t+5,split_t+6,gain_t+6,split_t+7,gain_t+7
3,dayofweek,694,622501.906511,707,637336.433803,608,637935.748361,611,633299.414748,620,635633.217813,657,638751.419492,615,638419.516815
4,is_off_day,149,324143.692828,190,309577.843413,158,302557.215340,150,306662.490749,159,305779.771414,168,303493.134174,161,301572.493967
2,day,772,59530.389652,830,64474.031361,668,58283.011086,660,58931.740629,693,59193.132679,745,61400.302596,675,58718.769853
24,rolling_std_28_s1,256,52061.756474,247,49406.681856,225,48439.817630,220,50194.816692,222,50098.817134,231,48159.859370,206,47945.180081
1,month,531,35857.349460,518,41262.919646,409,40108.901464,414,40243.697281,427,41229.214849,441,40791.467491,417,40865.547127
23,rolling_mean_28_s1,170,18509.643506,156,21006.579996,120,22913.417154,124,20636.678980,143,20665.055779,145,22349.908220,138,21105.882895
0,RECEIVER_SIGUNGU_NM,142,15308.483937,153,15743.839448,121,14305.477347,128,14783.115894,148,15193.007901,139,15610.036131,141,16037.625166
17,vol_lag_1,74,8028.933557,45,2315.758134,8,381.543297,1,15.553700,5,456.073498,4,221.364302,3,101.397203
25,ratio_lag_to_m28_s1,102,7874.606768,23,535.992878,11,541.191202,8,348.823801,22,1591.294403,11,844.641802,3,194.369701
11,pre_holiday_impact_score,179,7719.495283,188,7822.474656,151,7334.960110,149,7286.332693,156,7420.404206,165,7526.332182,154,7334.567517
